In [7]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
load_dotenv()
from pydantic import BaseModel, Field


In [2]:
model = ChatOpenAI(model='gpt-4o-mini')

In [5]:
class SentimentScchema(BaseModel):
    sentiment: Literal['positive','negative'] = Field(description='Sentiment of the text')

In [4]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX","Performance","Bug","Support"] = Field(description="the category of issue mention in the review")
    tone: Literal["angry","frustrated","disappointed","calm"] = Field(description="the emotional express by the user")
    urgency: Literal["low","medium","hight"] = Field(description="how urgent or critical issue appear to be")
    

In [6]:
structured_model = model.with_structured_output(SentimentScchema)
structred_mode2 = model.with_structured_output(DiagnosisSchema)


In [8]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal['positive','negative']
    diagnosis: dict
    response: str

In [9]:
def find_sentiment(state: ReviewState):
    prompt = f"for the following review find out sentiment \n{state['review']}"
    sentiment = structured_model.invoke(prompt).sentiment
    return {"sentiment": sentiment}

In [10]:
def check_sentiment(state: ReviewState)->Literal['positive_response','run_diagnosis']:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [16]:
def positive_response(state: ReviewState):
    prompt = f"Write a warm thank you message in response to this review:\n{state['review']}"
    response = model.invoke(prompt).content
    return {"response": response}


def run_diagnosis(state: ReviewState):
    prompt = f"""
    Diagnose this negative review:

    {state['review']}

    Return the issue_type, tone, urgency.
    """
    response = structred_mode2.invoke(prompt)
    return {"diagnosis": response.model_dump()}


def negative_response(state: ReviewState):
    prompt = f"""
    You are a support assistant.

    The user had a {state["diagnosis"]["issue_type"]} issue.
    The review sounded {state["diagnosis"]["tone"]}.
    The urgency is {state["diagnosis"]["urgency"]}.

    Write an empathetic, helpful resolution message.
    """
    
    response = model.invoke(prompt).content
    return {"response": response}

In [19]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment", find_sentiment)
graph.add_node("positive_response", positive_response)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)

graph.add_edge(START, "find_sentiment")

graph.add_conditional_edges(
    "find_sentiment",
    check_sentiment
)

graph.add_edge("positive_response", END)

graph.add_edge("run_diagnosis", "negative_response")

graph.add_edge("negative_response", END)

workflow = graph.compile()

In [22]:
initial_state = {
    'review':'nothing is working not getting otp'
}
workflow.invoke(initial_state)

{'review': 'nothing is working not getting otp',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'hight'},
 'response': "Subject: We're Here to Help!\n\nHi [User's Name],\n\nI sincerely apologize for the frustration you're experiencing with the bug issue. I understand how important it is to have everything functioning smoothly, especially when you're relying on it. \n\nTo help us resolve this as swiftly as possible, could you please provide any additional details about the issue? Specifically, if you could describe what you were doing when the bug occurred, any error messages you received, and the device or browser you are using, that would be incredibly helpful.\n\nRest assured, we are committed to resolving this for you as quickly as we can. Thank you for your patience, and I look forward to hearing from you soon so we can get this sorted out!\n\nBest regards,  \n[Your Name]  \n[Your Position]  \n[Your Contact Information]  "}